**FinTwitBERT + Classification Head**

***Architecture :***
```
Tweet → FinBERT → [CLS] token → Dropout → Dense(256) → Dropout → Dense(3)
```

**Import & Configuration**

In [3]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score
)

# ── Seeds reproductibilité ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path('data/processed')
RES_DIR    = Path('results')
MODEL_DIR  = Path('models/fintwit_classifier')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparamètres ──────────────────────────────────────────────────────────
BACKBONE     = 'ProsusAI/finbert'   # même tokenizer que le preprocessing (attention on a ajouter des tokens speciaux)
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

special_tokens = [f'TICKER_{s}' for s in [
    'AAPL','MSFT','GOOGL','AMZN','TSLA','META','NVDA','BRK','JPM',
    'JNJ','V','PG','UNH','HD','MA','PYPL','BAC','DIS','ADBE','NFLX',
    'CMCSA','XOM','VZ','INTC','T','CSCO','PFE'
]] + ['AMOUNT_']

tokenizer.add_tokens(special_tokens)

MAX_LENGTH   = 128
BATCH_SIZE   = 32
LR_BERT      = 2e-5    # learning rate faible pour le backbone pré-entraîné
LR_HEAD      = 1e-3    # learning rate plus élevé pour la tête de classification
WEIGHT_DECAY = 0.01
MAX_EPOCHS   = 10
PATIENCE     = 3       # early stopping sur val F1-macro
WARMUP_RATIO = 0.10    # 10% des steps = warmup linéaire

LABEL_NAMES = ['négatif', 'neutre', 'positif']

Device : cpu


**Chargement des données**

In [4]:
# ── Chargement des encodings des tweets pré-calculés (.pt) ───────────────────────────────

train_enc = torch.load(DATA_DIR / 'train_encodings.pt')
val_enc   = torch.load(DATA_DIR / 'val_encodings.pt')
test_enc  = torch.load(DATA_DIR / 'test_encodings.pt')

print(f'Train  : {train_enc["input_ids"].shape[0]} tweets')
print(f'Val    : {val_enc["input_ids"].shape[0]} tweets')
print(f'Test   : {test_enc["input_ids"].shape[0]} tweets')
print(f'Shape  : {train_enc["input_ids"].shape}  (N × max_length)')

Train  : 29576 tweets
Val    : 6338 tweets
Test   : 6338 tweets
Shape  : torch.Size([29576, 128])  (N × max_length)


In [5]:
# ── Dataset PyTorch ───────────────────────────────────────────────────────────
class TweetDataset(Dataset):
    """Encapsule les encodings pré-calculés dans un Dataset PyTorch."""
    def __init__(self, encodings):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = encodings['labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx]
        }

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = TweetDataset(train_enc)
val_ds   = TweetDataset(val_enc)
test_ds  = TweetDataset(test_enc)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

Batches — Train: 925 | Val: 199 | Test: 199


**Rappel de la baseline**

In [6]:
# ── Rappel des performances de la baseline TF-IDF + LogReg ───────────────────
with open(RES_DIR / 'metrics_baseline.json') as f:
    baseline = json.load(f)

print('=== BASELINE À BATTRE ===')

print(f'  Accuracy   : {baseline["val"]["accuracy"]:.4f}')
print(f'  F1-macro   : {baseline["val"]["f1_macro"]:.4f}')

print(f'  F1 négatif : {baseline["val"]["f1_negatif"]:.4f}')
print(f'  F1 neutre  : {baseline["val"]["f1_neutre"]:.4f}')
print(f'  F1 positif : {baseline["val"]["f1_positif"]:.4f}')

=== BASELINE À BATTRE ===
  Accuracy   : 0.7084
  F1-macro   : 0.6999
  F1 négatif : 0.6505
  F1 neutre  : 0.7179
  F1 positif : 0.7313


**Class weights**

In [7]:
# ── Class weights (calculés dans 4_preprocessing) ───────────────────────────
# Négatif sous-représenté (18.9%) → les erreurs ont un poids plus élevé
with open(RES_DIR / 'class_weights.json') as f:
    cw_data = json.load(f)

cw = cw_data['weights']  # {0: w0, 1: w1, 2: w2}
class_weights_tensor = torch.tensor(
    [cw['0'], cw['1'], cw['2']], dtype=torch.float
).to(DEVICE)

print('Class weights :', {LABEL_NAMES[i]: round(cw[str(i)], 3) for i in range(3)})

Class weights : {'négatif': 1.765, 'neutre': 0.763, 'positif': 0.891}


**Architecture du modèle**

**Pourquoi FinBERT et pas un BERT générique ?**

FinBERT (Yang et al., 2020) est pré-entraîné sur des textes financiers (rapports, news, tweets).  
Il connaît le vocabulaire spécifique : *bearish*, *rally*, *selloff*, *EPS*, *TICKER_XXX*.  
Un BERT générique traiterait ces termes comme des mots rares — FinBERT les comprend en contexte.

**Pourquoi le fine-tuning complet ?**  
À cette étape on veut mesurer le maximum de ce que le backbone peut donner.  
Les couches supérieures de BERT capturent la sémantique → elles doivent s'adapter à notre tâche.

In [8]:
class FinBertClassifier(nn.Module):
    """
    FinBERT + tête de classification 3 classes.
    
    Flux :
      input_ids, attention_mask
        → FinBERT encoder
        → CLS token [768]
        → Dropout(0.3)
        → Linear(768 → 256) + ReLU
        → Dropout(0.2)
        → Linear(256 → 3)
        → Softmax (via CrossEntropyLoss)
    """
    def __init__(self, backbone_name, num_classes=3, dropout1=0.3, dropout2=0.2):
        super().__init__()

        # Backbone pré-entraîné
        self.bert = AutoModel.from_pretrained(backbone_name)
        hidden_size = self.bert.config.hidden_size  # 768 pour FinBERT

        # Tête de classification
        self.drop1      = nn.Dropout(dropout1)
        self.dense      = nn.Linear(hidden_size, 256)
        self.relu       = nn.ReLU()
        self.drop2      = nn.Dropout(dropout2)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        # FinBERT encode la séquence
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # On prend uniquement le token [CLS] — représentation globale du tweet 
        cls_output = outputs.last_hidden_state[:, 0, :]  # shape : (batch, 768)

        # Tête de classification
        x = self.drop1(cls_output)
        x = self.relu(self.dense(x))
        x = self.drop2(x)
        logits = self.classifier(x)  # shape : (batch, 3)

        return logits


# ── Instanciation ─────────────────────────────────────────────────────────────
model = FinBertClassifier(BACKBONE).to(DEVICE)
model.bert.resize_token_embeddings(len(tokenizer))

# Compter les paramètres entraînables
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres totaux     : {total_params:,}')
print(f'Paramètres entraînables : {trainable_params:,}')

d:\M2_MoSEF\NLP_Project\NLP_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Juan David Alonso\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20911.62it/s]
BertModel LOAD REPORT from: 

Paramètres totaux     : 109,701,379
Paramètres entraînables : 109,701,379


**Optimizer, Scheduler, Loss**

In [9]:
# ── Learning rate différencié ─────────────────────────────────────────────────
# Backbone FinBERT : lr faible (2e-5) pour ne pas "oublier" le pré-entraînement
# Tête de classification : lr fort (1e-3) car initialisée aléatoirement

bert_params = list(model.bert.parameters())
head_params = [
    *model.dense.parameters(),
    *model.classifier.parameters()
]

optimizer = AdamW([
    {'params': bert_params, 'lr': LR_BERT},
    {'params': head_params, 'lr': LR_HEAD}
], weight_decay=WEIGHT_DECAY)

# ── Scheduler : warmup linéaire puis décroissance linéaire ───────────────────
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# ── Loss pondérée (class weights pour compenser le déséquilibre) ─────────────
# Négatif (18.9%) reçoit un poids plus élevé → le modèle est plus "pénalisé"
# quand il se trompe sur un tweet négatif
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

print(f'Total steps  : {total_steps}')
print(f'Warmup steps : {warmup_steps}')

Total steps  : 9250
Warmup steps : 925


**Entrainement du modèle**

In [10]:
def train_epoch(model, loader, optimizer, scheduler, criterion):
    """Une époque d'entraînement. Retourne loss moyenne."""
    model.train()
    total_loss = 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)

        loss.backward()

        # Gradient clipping : évite les explosions de gradient
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    """Évaluation sur val ou test. Retourne loss, accuracy, F1-macro, preds."""
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            probs  = torch.softmax(logits, dim=-1)

            total_loss  += loss.item()
            all_preds   += logits.argmax(dim=-1).cpu().tolist()
            all_labels  += labels.cpu().tolist()
            all_probs   += probs.cpu().tolist()

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, acc, f1_macro, all_preds, all_labels, all_probs

In [11]:
# ── Boucle principale ─────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

best_val_f1   = 0.0
patience_cpt  = 0
best_epoch    = 0
start_time    = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss  = train_epoch(model, train_loader, optimizer, scheduler, criterion)
    val_loss, val_acc, val_f1, _, _, _  = evaluate(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val Acc: {val_acc:.4f} | '
          f'Val F1: {val_f1:.4f} | '
          f'{elapsed:.0f}s')

    # ── Early stopping sur val F1-macro ───────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        best_epoch   = epoch
        patience_cpt = 0
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        print(f'  → Meilleur modèle sauvegardé (F1-macro val = {val_f1:.4f})')
    else:
        patience_cpt += 1
        if patience_cpt >= PATIENCE:
            print(f'\nEarly stopping à l\'époque {epoch} (patience={PATIENCE})')
            break

total_time = (time.time() - start_time) / 60
print(f'\nEntraînement terminé en {total_time:.1f} min')
print(f'Meilleure époque : {best_epoch} | Meilleur val F1-macro : {best_val_f1:.4f}')

d:\M2_MoSEF\NLP_Project\NLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

**Courbes d'apprentissage**

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='o')
axes[0].set_title('Loss')
axes[0].set_xlabel('Époque')
axes[0].legend()
axes[0].axvline(x=best_epoch, color='red', linestyle='--', label=f'Best epoch {best_epoch}')

# F1-macro
axes[1].plot(epochs_ran, history['val_f1'], label='Val F1-macro', marker='o', color='green')
axes[1].axhline(y=0.6993, color='red', linestyle='--', label='Baseline (0.6993)')
axes[1].set_title('Val F1-macro vs Baseline')
axes[1].set_xlabel('Époque')
axes[1].legend()

plt.tight_layout()
plt.savefig(RES_DIR / 'learning_curves_01.png', dpi=150)
plt.show()

**Evaluation finale sur le test**

In [ ]:
# ── Charger le meilleur modèle ────────────────────────────────────────────────
model.load_state_dict(torch.load(MODEL_DIR / 'best_model.pt', map_location=DEVICE))

# ── Évaluation sur test ───────────────────────────────────────────────────────
_, test_acc, test_f1, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion
)

# F1 par classe
f1_per_class = f1_score(test_labels, test_preds, average=None)

# ROC-AUC (one-vs-rest)
roc_auc = roc_auc_score(
    test_labels, test_probs, multi_class='ovr', average='macro'
)

print('=== RÉSULTATS TEST — 01 FinBERT Classifier ===')
print(f'  Accuracy  : {test_acc:.4f}')
print(f'  F1-macro  : {test_f1:.4f}')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print(f'  F1 négatif : {f1_per_class[0]:.4f}')
print(f'  F1 neutre  : {f1_per_class[1]:.4f}')
print(f'  F1 positif : {f1_per_class[2]:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=LABEL_NAMES))

**Matrice de confusion**

In [ ]:
cm = confusion_matrix(test_labels, test_preds, normalize='true')

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES
)
plt.title('Matrice de confusion — 01 FinBERT (test, normalisée)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig(RES_DIR / 'confusion_matrix_01.png', dpi=150)
plt.show()

**Sauvegarde des métriques**

In [ ]:
metrics_01 = {
    'model':                '01_finbert_classifier',
    'accuracy':             round(test_acc, 4),
    'f1_macro':             round(test_f1, 4),
    'f1_per_class': {
        'negatif':          round(float(f1_per_class[0]), 4),
        'neutre':           round(float(f1_per_class[1]), 4),
        'positif':          round(float(f1_per_class[2]), 4),
    },
    'roc_auc':              round(roc_auc, 4),
    'training_time_minutes': round(total_time, 1),
    'best_epoch':           best_epoch,
    'trainable_params':     trainable_params,
    'val_f1_history':       [round(v, 4) for v in history['val_f1']],
}

with open(RES_DIR / 'metrics_01.json', 'w') as f:
    json.dump(metrics_01, f, indent=2)

print('Métriques sauvegardées : results/metrics_01.json')
print(json.dumps(metrics_01, indent=2))